# 09. 모델 계열 변경: LR → XGBoost (H8)

## 배경
- LR 계열은 피처를 바꿔도 LB 0.270~0.274에서 정체 (05, 07 E, 07 F)
- 07 확인 1: stats를 넣으면 LR은 test에서 변이 많은 클래스로 쏠림 → 선형 합산 구조가 원인일 가능성(로지스틱회귀라서 점수가 정체이지 않을까)

## 가설
- H8: 트리 모델은 피처를 합산하지 않고 분기로 판단하므로, 같은 피처(any + stats)에서 LB가 LR보다 높다
- H0: 모델을 바꿔도 LB는 0.27 근처에 머문다

## 검증 방법
- 피처: 07 B 조합(any + stats), 모델만 교체
- 하이퍼파라미터: 02와 같은 제공 베이스라인 설정 (튜닝은 10에서)
- 클래스 불균형: balanced sample_weight (LR의 class_weight='balanced'와 같은 역할)
- 판단 기준: LB가 05 최고값 0.2739보다 오르면 H8 채택

In [ ]:
# 09. H8: 트리 모델(XGBoost)은 LR보다 test 분포 차이에 덜 민감하다
import sys; sys.path.append('..')
import pandas as pd, xgboost as xgb
from sklearn.utils.class_weight import compute_sample_weight
from src.config import SEED
from src.data import load_data, get_gene_cols
from src.features import mutation_features
from src.validation import seed_everything, encode_target, run_cv, save_submission

seed_everything()
train, test, submission = load_data()
genes = get_gene_cols(train)
y, le = encode_target(train)
F_tr, F_te = mutation_features(train, genes), mutation_features(test, genes)

# XGBoost는 class_weight가 없어서, fit할 때 balanced 가중치를 자동으로 넣는 래퍼를 둠
class BalancedXGB(xgb.XGBClassifier):
    def fit(self, X, y, **kw):
        return super().fit(X, y, sample_weight=compute_sample_weight('balanced', y), **kw)

make_xgb = lambda: BalancedXGB(n_estimators=100, learning_rate=0.1, max_depth=6,
                            tree_method='hist', eval_metric='mlogloss',
                            random_state=SEED, n_jobs=-1)

X = pd.concat([F_tr['any'], F_tr['stats']], axis=1)
X_test = pd.concat([F_te['any'], F_te['stats']], axis=1)
r = run_cv(make_xgb, X, y, X_test)
save_submission(submission, r['test_proba'], le, name='xgb_any_stats', cv_score=r['cv_mean'])

[fold 0] Macro F1 = 0.3963
[fold 1] Macro F1 = 0.3928
[fold 2] Macro F1 = 0.3858
[fold 3] Macro F1 = 0.3809
[fold 4] Macro F1 = 0.3837
CV Macro F1 = 0.3879 ± 0.0058 (OOF 0.3889)
저장 완료: C:\Users\USER\Desktop\hackathon\submissions\1110_xgb_any_stats_0.388.csv


WindowsPath('C:/Users/USER/Desktop/hackathon/submissions/1110_xgb_any_stats_0.388.csv')